# Vídeo 2 – Demonstração Consolidada (Atualizado)
Sem `LLMChain` e sem `.run()`. Agora usamos `prompt | llm` e `.invoke()`.

Instale:
```
pip install -U python-dotenv openai langchain langchain-openai langchain-community langgraph faiss-cpu typing_extensions
```
Crie `.env`:
```
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o-mini
```


## Parte 1 – LangChain isolado (pipeline moderno)

In [1]:
"""
Vídeo 2 – Parte 1: LangChain isolado (atualizado)
Agora usando RunnableSequence (`prompt | llm`) + `.invoke()`
em vez de LLMChain.run().

Dependências:
    pip install -U python-dotenv openai langchain langchain-openai langchain-community faiss-cpu

.env (na mesma pasta):
    OPENAI_API_KEY=sk-...
    OPENAI_MODEL=gpt-4o-mini   # opcional
"""

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS

# Configuração
load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0.2, max_tokens=128)

# 1) PromptTemplate + LLM (pipeline moderno)
template = PromptTemplate.from_template("Resuma em uma frase: {texto}")
chain = template | llm

resumo = chain.invoke({"texto": "LangGraph organiza fluxos de IA como grafos de execução."})
print("Resumo (LangChain isolado):", resumo.content)

# 2) VectorStore (isolado) com FAISS
embeddings = OpenAIEmbeddings()
docs = [
    "LangGraph cria nós e arestas para orquestrar agentes.",
    "LangChain dá ferramentas para prompts e integrações.",
    "Grafos permitem rotas condicionais explícitas."
]
vectorstore = FAISS.from_texts(docs, embeddings)

query = "Como o LangGraph organiza fluxos?"
docs_retrieved = vectorstore.similarity_search(query, k=1)
print("Busca no VectorStore (LangChain isolado):", docs_retrieved[0].page_content)


Resumo (LangChain isolado): LangGraph estrutura fluxos de inteligência artificial em forma de grafos de execução.
Busca no VectorStore (LangChain isolado): LangGraph cria nós e arestas para orquestrar agentes.


## Parte 2 – LangGraph (componentes como nós)

In [2]:
"""
Vídeo 2 – Parte 2: LangGraph (atualizado)
Usa PromptTemplate | llm + .invoke() dentro dos nós.

Dependências:
    pip install -U python-dotenv openai langchain langchain-openai langchain-community langgraph faiss-cpu typing_extensions

.env (na mesma pasta):
    OPENAI_API_KEY=sk-...
    OPENAI_MODEL=gpt-4o-mini   # opcional
"""

import os
from typing_extensions import TypedDict, Annotated
from typing import List
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS

# Configuração
load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0.2, max_tokens=128)
embeddings = OpenAIEmbeddings()

# Estado compartilhado
class State(TypedDict, total=False):
    texto: str
    resumo: str
    docs: List[str]
    consulta: str
    resposta: str
    _vs: object
    historico: Annotated[list, add_messages]

# Nó 1: resumo
def resumir_node(state: State) -> State:
    template = PromptTemplate.from_template("Resuma em uma frase: {texto}")
    chain = template | llm
    resposta = chain.invoke({"texto": state["texto"]})
    return {"resumo": resposta.content, "historico": [{"role": "system", "content": "resumir_node"}]}

# Nó 2: indexação
def indexar_docs_node(state: State) -> State:
    vs = FAISS.from_texts(state["docs"], embeddings)
    return {"_vs": vs, "historico": [{"role": "system", "content": "indexar_docs_node"}]}

# Nó 3: busca
def buscar_node(state: State) -> State:
    vs = state.get("_vs")
    if not vs:
        return {"resposta": "VectorStore não encontrado.", "historico": [{"role": "system", "content": "buscar_node (sem VS)"}]}
    result = vs.similarity_search(state["consulta"], k=1)[0].page_content
    return {"resposta": result, "historico": [{"role": "system", "content": "buscar_node"}]}

# Construção do grafo
grafo = StateGraph(State)
grafo.add_node("resumir", resumir_node)
grafo.add_node("indexar_docs", indexar_docs_node)
grafo.add_node("buscar", buscar_node)

grafo.set_entry_point("resumir")
grafo.add_edge("resumir", "indexar_docs")
grafo.add_edge("indexar_docs", "buscar")
grafo.add_edge("buscar", END)

app = grafo.compile()

if __name__ == "__main__":
    print("=== Estrutura do grafo (ASCII real) ===")
    print(app.get_graph().draw_ascii())

    estado_inicial: State = {
        "texto": "LangGraph organiza fluxos de agentes como grafos de execução.",
        "docs": [
            "LangGraph cria nós e arestas para orquestrar agentes.",
            "LangChain dá ferramentas para prompts e integrações.",
            "Grafos permitem rotas condicionais explícitas."
        ],
        "consulta": "Como o LangGraph organiza fluxos?",
    }

    print("\n=== Execução ===")
    print(app.invoke(estado_inicial))


=== Estrutura do grafo (ASCII real) ===
  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
  +---------+    
  | resumir |    
  +---------+    
        *        
        *        
        *        
+--------------+ 
| indexar_docs | 
+--------------+ 
        *        
        *        
        *        
   +--------+    
   | buscar |    
   +--------+    
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    

=== Execução ===
{'texto': 'LangGraph organiza fluxos de agentes como grafos de execução.', 'resumo': 'LangGraph estrutura fluxos de agentes em forma de grafos de execução.', 'docs': ['LangGraph cria nós e arestas para orquestrar agentes.', 'LangChain dá ferramentas para prompts e integrações.', 'Grafos permitem rotas condicionais explícitas.'], 'consulta': 'Como o LangGraph organiza fluxos?', 'resposta': 'LangGraph cria nós e arestas para orquestrar agentes.', '_vs': 

In [3]:
# Execução resumida da Parte 2
print(app.get_graph().draw_ascii())
estado_inicial = {
    'texto': 'LangGraph organiza fluxos de agentes como grafos de execução.',
    'docs': [
        'LangGraph cria nós e arestas para orquestrar agentes.',
        'LangChain dá ferramentas para prompts e integrações.',
        'Grafos permitem rotas condicionais explícitas.'
    ],
    'consulta': 'Como o LangGraph organiza fluxos?',
}
print(app.invoke(estado_inicial))


  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
  +---------+    
  | resumir |    
  +---------+    
        *        
        *        
        *        
+--------------+ 
| indexar_docs | 
+--------------+ 
        *        
        *        
        *        
   +--------+    
   | buscar |    
   +--------+    
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    
{'texto': 'LangGraph organiza fluxos de agentes como grafos de execução.', 'resumo': 'LangGraph estrutura fluxos de agentes em forma de grafos de execução.', 'docs': ['LangGraph cria nós e arestas para orquestrar agentes.', 'LangChain dá ferramentas para prompts e integrações.', 'Grafos permitem rotas condicionais explícitas.'], 'consulta': 'Como o LangGraph organiza fluxos?', 'resposta': 'LangGraph cria nós e arestas para orquestrar agentes.', '_vs': <langchain_community.vectorstores.faiss.FAISS object at 0x